# Preparação dos Dados - FlowCarreiras

Este notebook documenta e valida a preparação das duas bases abertas utilizadas no projeto:

- **Mapa Cultural de Pernambuco:** recorte de 1.000 perfis individuais com área artística/criativa declarada, extraídos da API pública.
- **contempArt:** 441 artistas em início de carreira ligados a escolas de arte alemãs.

As bases são complementares, mas possuem contextos e unidades de análise diferentes. Portanto, são preparadas e analisadas separadamente.

## 1. Configuração e caminhos

Os scripts reproduzíveis de extração, limpeza e criação de variáveis estão em `analise-dados/src/`. Este notebook utiliza os arquivos gerados por esses scripts para demonstrar e validar o processo.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 100)

cwd = Path.cwd()
if (cwd / "analise-dados").exists():
    DATA_DIR = cwd / "analise-dados" / "data"
elif cwd.name == "notebooks":
    DATA_DIR = cwd.parent / "data"
else:
    DATA_DIR = cwd / "data"

RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
print(f"Diretório de dados: {DATA_DIR.resolve()}")

Diretório de dados: C:\Users\ganto\OneDrive\Área de Trabalho\FlowCarreiras\analise-dados\data


## 2. Metadados da extração do Mapa Cultural PE

O JSON bruto preserva a fonte, os parâmetros, o horário da extração e as páginas consultadas. Isso permite auditar e reproduzir o recorte utilizado.

In [2]:
mapa_raw_path = RAW_DIR / "mapa_cultural_pe_agentes.json"
mapa_raw = json.loads(mapa_raw_path.read_text(encoding="utf-8"))

metadados_extracao = pd.DataFrame({
    "informacao": ["fonte", "extraido_em_utc", "total_registros", "paginas_consultadas", "criterio_local"],
    "valor": [
        mapa_raw["fonte"],
        mapa_raw["extraido_em_utc"],
        mapa_raw["total_registros"],
        len(mapa_raw["urls_consultadas"]),
        mapa_raw["consulta"]["criterio_local"],
    ],
})
display(metadados_extracao)

,informacao,valor
0,fonte,https://www.mapacultural.pe.gov.br/api/agent/find
1,extraido_em_utc,2026-06-09T20:58:30.650939+00:00
2,total_registros,1000
3,paginas_consultadas,17
4,criterio_local,perfil individual com pelo menos uma area artistica/criativa e sem marcadores evidentes de teste...


## 3. Carregamento das bases limpas e enriquecidas

A versão **limpa** preserva os dados tratados próximos às fontes. A versão **enriquecida** adiciona variáveis derivadas criadas pelo grupo, sem substituir os arquivos limpos.

In [3]:
mapa_limpo = pd.read_csv(PROCESSED_DIR / "mapa_cultural_pe_agentes.csv")
mapa_enriquecido = pd.read_csv(PROCESSED_DIR / "mapa_cultural_pe_agentes_enriquecido.csv")
contempart_limpo = pd.read_csv(PROCESSED_DIR / "contempart_artists.csv")
contempart_enriquecido = pd.read_csv(PROCESSED_DIR / "contempart_artists_enriquecido.csv")

resumo_bases = pd.DataFrame([
    ["Mapa Cultural PE - limpo", *mapa_limpo.shape],
    ["Mapa Cultural PE - enriquecido", *mapa_enriquecido.shape],
    ["contempArt - limpo", *contempart_limpo.shape],
    ["contempArt - enriquecido", *contempart_enriquecido.shape],
], columns=["base", "linhas", "colunas"])
display(resumo_bases)

,base,linhas,colunas
0,Mapa Cultural PE - limpo,1000,12
1,Mapa Cultural PE - enriquecido,1000,21
2,contempArt - limpo,441,25
3,contempArt - enriquecido,441,33


## 4. Validação de identificadores e duplicidades

Cada base possui um identificador próprio. Duplicidades são verificadas separadamente por `id` e `artist_id`.

In [4]:
validacao_ids = pd.DataFrame([
    ["Mapa Cultural PE", len(mapa_limpo), mapa_limpo["id"].nunique(), mapa_limpo["id"].duplicated().sum()],
    ["contempArt", len(contempart_limpo), contempart_limpo["artist_id"].nunique(), contempart_limpo["artist_id"].duplicated().sum()],
], columns=["base", "registros", "identificadores_unicos", "duplicidades"])
display(validacao_ids)

,base,registros,identificadores_unicos,duplicidades
0,Mapa Cultural PE,1000,1000,0
1,contempArt,441,441,0


## 5. Valores ausentes

Valores ausentes são mantidos como ausentes. Eles não são substituídos por zero, média ou categorias inventadas, pois isso poderia distorcer as análises.

In [5]:
def resumo_ausentes(dataframe):
    total = dataframe.isna().sum()
    percentual = (total / len(dataframe) * 100).round(2)
    return pd.DataFrame({"ausentes": total, "percentual": percentual}).sort_values("percentual", ascending=False)

print("Mapa Cultural PE")
display(resumo_ausentes(mapa_limpo))

print("contempArt")
display(resumo_ausentes(contempart_limpo))

Mapa Cultural PE


,ausentes,percentual
termos_etnias,1000,100.0
termos_funcoes,894,89.4
termos_tags,718,71.8
termos_subareas,626,62.6
descricao_curta,3,0.3
tipo_id,0,0.0
id,0,0.0
nome,0,0.0
termos_areas,0,0.0
tipo_nome,0,0.0


contempArt


,ausentes,percentual
avg_likes,241,54.65
avg_comments,241,54.65
region,208,47.17
country_iso3,208,47.17
continent,208,47.17
website,201,45.58
is_business,82,18.59
following_count,82,18.59
posts_count,82,18.59
follower_count,82,18.59


## 6. Conversão e validação das datas

As datas são convertidas para `datetime`. Valores inválidos se tornam ausentes e atualizações anteriores à criação são contabilizadas como inconsistências.

In [6]:
datas = mapa_limpo[["data_criacao", "data_atualizacao"]].copy()
datas["data_criacao"] = pd.to_datetime(datas["data_criacao"], errors="coerce")
datas["data_atualizacao"] = pd.to_datetime(datas["data_atualizacao"], errors="coerce")

validacao_datas = pd.DataFrame({
    "metrica": ["datas de criação inválidas/ausentes", "datas de atualização inválidas/ausentes", "atualizações anteriores à criação"],
    "quantidade": [
        datas["data_criacao"].isna().sum(),
        datas["data_atualizacao"].isna().sum(),
        (datas["data_atualizacao"] < datas["data_criacao"]).sum(),
    ],
})
display(validacao_datas)

,metrica,quantidade
0,datas de criação inválidas/ausentes,0
1,datas de atualização inválidas/ausentes,0
2,atualizações anteriores à criação,0


## 7. Listas de áreas e tags

As listas do Mapa Cultural PE foram padronizadas com o separador ` | `. A separação em múltiplas linhas deve ocorrer apenas durante análises de frequência.

In [7]:
amostra_termos = mapa_limpo.loc[
    mapa_limpo["termos_areas"].notna(),
    ["nome", "termos_areas", "termos_tags", "termos_subareas"],
].head(10)
display(amostra_termos)

,nome,termos_areas,termos_tags,termos_subareas
0,Flávio Barbosa da Silva,Design | Gestão Cultural | Patrimônio Cultural,NaN,Design
1,Janaína Branco,Design | Design de Moda | Economia Criativa e da Cultura | Gestão Cultural,Fundarpe | Secult,Design | Design da Informação | Design de Produto | UX Design
2,Juliana Rezende,Artesanato | Cultura e Turismo | Gestão Cultural,NaN,NaN
3,José Neto Barbosa,Cinema | Cultura LGBTQIAPN+ | Outra | Produção Cultural | Teatro | Televisão,NaN,Audiovisual | Teatro Adulto
4,clarice andrade,Audiovisual e Mídias Interativas | Comunicação | Cultura e Educação | Gestão Cultural | Museu (P...,Cais do Sertão | artes visuais | gestao cultural | museu,NaN
5,Manoela Antunes,Leitura | Literatura | Livro,NaN,NaN
6,Milena Evangelista,Audiovisual e Mídias Interativas | Comunicação | Gestão Cultural,NaN,NaN
7,Marcela Wanderley,Artes Visuais | Artesanato | Gestão Cultural | Museu (Patrimônio Material) | Produção Cultural,NaN,Cadeia Criativa
8,Felipe Cabral,Arte Digital,NaN,NaN
9,Jacira França,Culturas Populares | Gestão Cultural | Patrimônio Cultural Imaterial | Sociologia,NaN,Cultura Popular


## 8. Variáveis derivadas

As variáveis derivadas transformam dados existentes em indicadores úteis para EDA, regressão, classificação e dashboard.

In [8]:
derivadas_mapa = ["possui_descricao", "possui_tags", "possui_funcoes", "quantidade_areas", "perfil_multidisciplinar", "perfil_minimamente_estruturado"]
derivadas_contempart = ["possui_instagram", "possui_website", "somente_instagram_informado", "sem_presenca_digital_informada", "taxa_engajamento", "quadrante_imagens_visibilidade"]

print("Mapa Cultural PE - variáveis derivadas")
display(mapa_enriquecido[["id", "nome", *derivadas_mapa]].head(10))

print("contempArt - variáveis derivadas")
display(contempart_enriquecido[["artist_id", "full_name", *derivadas_contempart]].head(10))

Mapa Cultural PE - variáveis derivadas


,id,nome,possui_descricao,possui_tags,possui_funcoes,quantidade_areas,perfil_multidisciplinar,perfil_minimamente_estruturado
0,7,Flávio Barbosa da Silva,True,False,False,3,True,False
1,17,Janaína Branco,True,True,False,4,True,True
2,18,Juliana Rezende,True,False,False,3,True,False
3,19,José Neto Barbosa,True,False,False,6,True,False
4,21,clarice andrade,True,True,False,6,True,True
5,22,Manoela Antunes,True,False,False,3,True,False
6,24,Milena Evangelista,True,False,False,3,True,False
7,25,Marcela Wanderley,True,False,False,5,True,False
8,28,Felipe Cabral,True,False,False,1,False,False
9,49,Jacira França,True,False,False,4,True,False


contempArt - variáveis derivadas


,artist_id,full_name,possui_instagram,possui_website,somente_instagram_informado,sem_presenca_digital_informada,taxa_engajamento,quadrante_imagens_visibilidade
0,agneswrobel,Agnes Wrobel,False,True,False,False,NaN,NaN
1,albertgouthier,Albert Gouthier,True,False,True,False,10.461327,volume_alto_visibilidade_alta
2,alessiaschuth,Alessia Schuth,False,True,False,False,NaN,NaN
3,alexandermick,Alexander Mick,True,True,False,False,21.600000,volume_baixo_visibilidade_alta
4,alexanderschulz,Alexander Schulz,True,True,False,False,NaN,volume_baixo_visibilidade_alta
5,alexiatrawinski,ALEXIA TRAWINSKI,True,True,False,False,23.188406,volume_alto_visibilidade_baixa
6,alicegericke,Alice Gericke,True,False,True,False,23.155738,volume_baixo_visibilidade_baixa
7,alidawarzecha,Alida Warzecha,True,False,True,False,21.397849,volume_baixo_visibilidade_alta
8,alineschwibbe,Aline Schwibbe,True,True,False,False,NaN,volume_baixo_visibilidade_alta
9,alisarmaya,Alisar Maya,True,False,True,False,7.028388,volume_baixo_visibilidade_alta


In [9]:
resumo_derivadas = pd.DataFrame({
    "metrica": [
        "Mapa PE: percentual com descrição",
        "Mapa PE: mediana de áreas por agente",
        "contempArt: percentual com Instagram",
        "contempArt: percentual com website",
        "Mapa PE: percentual de perfis minimamente estruturados",
        "contempArt: percentual somente com Instagram informado",
        "contempArt: mediana da taxa de engajamento calculável",
    ],
    "valor": [
        f"{mapa_enriquecido['possui_descricao'].mean() * 100:.2f}%",
        mapa_enriquecido["quantidade_areas"].median(),
        f"{contempart_enriquecido['possui_instagram'].mean() * 100:.2f}%",
        f"{contempart_enriquecido['possui_website'].mean() * 100:.2f}%",
        f"{mapa_enriquecido['perfil_minimamente_estruturado'].mean() * 100:.2f}%",
        f"{contempart_enriquecido['somente_instagram_informado'].mean() * 100:.2f}%",
        contempart_enriquecido["taxa_engajamento"].median(),
    ],
})
display(resumo_derivadas)

,metrica,valor
0,Mapa PE: percentual com descrição,99.70%
1,Mapa PE: mediana de áreas por agente,2.0
2,contempArt: percentual com Instagram,82.99%
3,contempArt: percentual com website,54.42%
4,Mapa PE: percentual de perfis minimamente estruturados,33.60%
5,contempArt: percentual somente com Instagram informado,42.63%
6,contempArt: mediana da taxa de engajamento calculável,14.357876


## 9. Checklist final da preparação

- Bases brutas preservadas para auditoria.
- Bases limpas mantidas separadamente.
- Identificadores únicos e duplicidades verificados.
- Ausentes preservados e documentados.
- Datas convertidas e inconsistências temporais verificadas.
- Listas de áreas e tags padronizadas.
- Variáveis derivadas geradas em arquivos enriquecidos separados.
- Bases prontas para os notebooks de EDA, modelagem e dashboard.

### Limitação principal

O Mapa Cultural PE utiliza os primeiros 1.000 perfis individuais que atendem ao critério artístico/criativo documentado. Como a API não possui um campo que comprove a profissão artística, esse filtro é uma aproximação reproduzível e o recorte não deve ser tratado como amostra aleatória representativa dos mais de 61 mil agentes cadastrados.